# Stage A2 — CKA analysis

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
Computes **Linear CKA** between every layer pair of the two models. CKA
compares the *pairwise-similarity structure* of the representations (the
Gram matrix "fingerprint"), and is invariant to rotations, permutations and
isotropic scaling — exactly the symmetries under which a network's function
is preserved. So it is blind to meaningless coordinate differences and
sensitive to what is actually near what.

## How to read the result
- **Hypothesis supported:** the trained-vs-trained matrix shows a *hot
  diagonal* (early layers match early, middle match middle), while the
  trained-vs-random matrix is uniformly cold.
- **Hypothesis rejected:** both matrices look like structureless noise.

## Success criterion
Diagonal CKA mean > 0.5 with the random baseline < ~0.15 (a 3-4x gap).


In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt

In [ ]:
# Functions
def linear_cka(X, Y):
    """X: [n, d1], Y: [n, d2]. Returns scalar in [0, 1]."""
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    xty = X.T @ Y
    num = (xty ** 2).sum()
    den = np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)
    return float(num / den)


def cka_matrix(A, B):
    M = np.zeros((A.shape[0], B.shape[0]))
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            M[i, j] = linear_cka(A[i], B[j])
    return M


def main():
    data = np.load(str(DATA_DIR / "activations.npz"))
    A, B, R = data["A_layers"], data["B_layers"], data["R_layers"]

    print("Computing CKA: GPT-2 vs Pythia...")
    M_trained = cka_matrix(A, B)
    print("Computing CKA baseline: GPT-2 vs random-weights Pythia...")
    M_random = cka_matrix(A, R)

    diag = np.diag(M_trained[: min(M_trained.shape), : min(M_trained.shape)])
    print(f"\nTrained-vs-trained: mean={M_trained.mean():.3f}, "
          f"diag mean={diag.mean():.3f}, max={M_trained.max():.3f}")
    print(f"Trained-vs-random:  mean={M_random.mean():.3f}, "
          f"max={M_random.max():.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, M, title in [
        (axes[0], M_trained, "GPT-2 vs Pythia-160M (both trained)"),
        (axes[1], M_random, "GPT-2 vs Pythia (random weights)"),
    ]:
        im = ax.imshow(M, vmin=0, vmax=1, cmap="magma", origin="lower")
        ax.set_xlabel("Pythia layer")
        ax.set_ylabel("GPT-2 layer")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(str(DATA_DIR / "cka_matrix.png"), dpi=150)
    # cache the matrices so A2-bis can analyse them without recomputing
    np.savez_compressed(str(DATA_DIR / "cka_matrices.npz"),
                        trained=M_trained, random=M_random)
    print("\nSaved cka_matrix.png")
    print("Hypothesis supported if left panel shows a hot diagonal "
          "and right panel is uniformly cold.")

In [ ]:
# Run the analysis (requires activations.npz from stage A1)
main()

## A2-bis — Measuring the structure, not eyeballing it

Five blocks of output, in order of how much they can be argued with:

1. **Contrast against the control** — the headline; needs no interpretation.
2. **Depth trend on the diagonal** — Spearman rho over the longest rising
   run. Requires no normalization, so it is the most defensible
   depth-ordering evidence available. Compare it with A3's stitching curve.
3. **Is it a diagonal?** — row argmax, offsets, diagonal/off-diagonal
   ratio, proximity correlation, permutation test.
4. **Hubness** — row and column means plus the rank-1 share. A large
   rank-1 share means the bright bands are marginal effects (some layers
   are generically similar to everything), not genuine pairings.
5. **Lift** — the same correspondence with marginals divided out, which
   is the standard correction for exactly that effect.

Calibrated on synthetic controls: a constructed diagonal gives mean
offset 0.00, ratio 2.55, corr +0.83; a constructed block pattern gives
6.00, 1.12, −0.08.

Saves `cka_real_structure.png` (measured matrix · depth trend · lift).

In [ ]:
# A2-bis: measure the structure rather than eyeballing it
from scipy.stats import spearmanr

p_cache = DATA_DIR / "cka_matrices.npz"
if p_cache.exists():
    d = np.load(str(p_cache))
    M_t, M_r = d["trained"], d["random"]
    print("loaded cached CKA matrices")
else:
    print("no cache found - recomputing (a few minutes)...")
    data = np.load(str(DATA_DIR / "activations.npz"))
    A, B, R = data["A_layers"], data["B_layers"], data["R_layers"]
    M_t, M_r = cka_matrix(A, B), cka_matrix(A, R)
    np.savez_compressed(str(p_cache), trained=M_t, random=M_r)

L = M_t.shape[0]
idx = np.arange(L)
ii, jj = np.meshgrid(idx, np.arange(M_t.shape[1]), indexing="ij")
diag = np.diag(M_t)
offdiag = M_t[~np.eye(L, dtype=bool)].mean()

print("\n" + "=" * 62)
print("1. CONTRAST AGAINST THE CONTROL  (the headline result)")
print("=" * 62)
print(f"  trained mean {M_t.mean():.3f}   random mean {M_r.mean():.3f}"
      f"   ratio {M_t.mean() / M_r.mean():.1f}x")
print(f"  trained diag {diag.mean():.3f}   random max  {M_r.max():.3f}"
      f"   ratio {diag.mean() / M_r.mean():.1f}x")

print("\n" + "=" * 62)
print("2. DEPTH TREND ON THE DIAGONAL  (needs no correction)")
print("=" * 62)
for k in range(L):
    print(f"  L{k:2d}: {diag[k]:.3f}  " + "#" * int(diag[k] * 40))
best = (None, -2, 1.0)
for lo in range(0, L - 4):
    for hi in range(lo + 4, L):
        seg = diag[lo:hi + 1]
        rho, pv = spearmanr(np.arange(len(seg)), seg)
        if rho > best[1] or (rho == best[1] and hi - lo >
                             (best[0][1] - best[0][0] if best[0] else 0)):
            best = ((lo, hi), rho, pv)
(lo, hi), rho, pv = best
mono = all(diag[k] < diag[k + 1] for k in range(lo, hi))
rho_all, p_all = spearmanr(idx, diag)
print(f"\n  longest strong run: L{lo}-L{hi}  "
      f"rho = {rho:.3f}, p = {pv:.6f}"
      f"{'  (strictly monotone)' if mono else ''}")
print(f"  values {diag[lo]:.3f} -> {diag[hi]:.3f}")
print(f"  all 13 layers: rho = {rho_all:.3f}, p = {p_all:.4f}")
print("  compare with the A3 stitching curve: if both rise over the same")
print("  span, two independent measures agree on depth ordering.")

print("\n" + "=" * 62)
print("3. IS IT A DIAGONAL?  (raw matrix)")
print("=" * 62)
argmax = M_t.argmax(1)
offset = np.abs(argmax - idx)
corr = float(np.corrcoef(M_t.ravel(), -np.abs(ii - jj).ravel())[0, 1])
rng = np.random.default_rng(0)
null = np.array([M_t[idx, rng.permutation(L)].mean() for _ in range(5000)])
pval = float((null >= diag.mean()).mean())
print(f"  best match per GPT-2 layer : {argmax}")
print(f"  offset from the diagonal   : {offset}")
print(f"  mean |offset| {offset.mean():.2f}   within 2 layers "
      f"{(offset <= 2).mean():.2f}")
print(f"  diagonal {diag.mean():.3f} vs off-diagonal {offdiag:.3f}   "
      f"(ratio {diag.mean() / offdiag:.2f})")
print(f"  corr(similarity, proximity to diagonal) : {corr:+.3f}")
print(f"  permutation test on the diagonal        : p = {pval:.4f}")

print("\n" + "=" * 62)
print("4. HUBNESS  (why the raw panel looks blocky)")
print("=" * 62)
rm, cm = M_t.mean(1), M_t.mean(0)
print(f"  GPT-2 row means   : {np.round(rm, 3)}")
print(f"    range {rm.min():.3f} - {rm.max():.3f}  "
      f"({rm.max() / rm.min():.1f}x spread)")
print(f"  Pythia col means  : {np.round(cm, 3)}")
print(f"    range {cm.min():.3f} - {cm.max():.3f}  "
      f"({cm.max() / cm.min():.1f}x spread)")
s = np.linalg.svd(M_t, compute_uv=False)
print(f"  rank-1 share of matrix energy: "
      f"{100 * s[0] ** 2 / (s ** 2).sum():.1f}%")
print("  a large rank-1 share means some layers are generically similar")
print("  to everything - the bands are marginal effects, not pairings.")

print("\n" + "=" * 62)
print("5. CORRESPONDENCE WITH MARGINALS DIVIDED OUT  (lift)")
print("=" * 62)
lift = M_t / (np.outer(rm, cm) / M_t.mean())
am_l = lift.argmax(1)
off_l = np.abs(am_l - idx)
dl = np.diag(lift)
odl = lift[~np.eye(L, dtype=bool)].mean()
null_l = np.array([lift[idx, rng.permutation(L)].mean()
                   for _ in range(5000)])
p_l = float((null_l >= dl.mean()).mean())
corr_l = float(np.corrcoef(lift.ravel(), -np.abs(ii - jj).ravel())[0, 1])
print(f"  best match per layer : {am_l}")
print(f"  offset               : {off_l}")
print(f"  mean |offset| {off_l.mean():.2f} (raw was {offset.mean():.2f})"
      f"   within 2: {(off_l <= 2).mean():.2f} "
      f"(raw {(offset <= 2).mean():.2f})")
print(f"  diagonal lift {dl.mean():.3f} vs off-diagonal {odl:.3f}   "
      f"p = {p_l:.4f}")
print(f"  corr(lift, proximity) {corr_l:+.3f} "
      f"(raw {corr:+.3f})")
print("\n  per-layer diagonal enrichment (>1 = above expectation):")
for k in range(L):
    mark = "  <--" if dl[k] > 1.05 else ("  (below)" if dl[k] < 0.95
                                         else "")
    print(f"    L{k:2d}: {dl[k]:.2f}{mark}")

print("\n" + "=" * 62)
print("VERDICT")
print("=" * 62)
strong = offset.mean() <= 2.0 and corr > 0.3 and pval < 0.01
depth = rho > 0.9 and pv < 0.01 and (hi - lo) >= 5
weak = diag.mean() / offdiag > 1.05 and pval < 0.05
if strong:
    print("  DIAGONAL SUPPORTED - layer-to-layer correspondence measured.")
elif depth and weak:
    print("  DEPTH-ORDERED, DIAGONAL ELEVATED, CORRESPONDENCE SOFT.")
    print(f"  Defensible wording: the diagonal is elevated (ratio "
          f"{diag.mean() / offdiag:.2f}, p = {pval:.3f}) and rises")
    print(f"  monotonically across L{lo}-L{hi} (rho = {rho:.3f}, "
          f"p < {max(pv, 1e-5):.5f}); the raw matrix is dominated by")
    print(f"  hubness ({100 * s[0] ** 2 / (s ** 2).sum():.0f}% rank-1), "
          f"and with marginals divided out the diagonal")
    print(f"  is enriched {dl.mean() / odl:.2f}x (p = {p_l:.3f}) with "
          f"{(off_l <= 2).mean():.0%} of layers matching within two.")
    print("  Do NOT call the raw panel a hot diagonal.")
elif weak:
    print("  DIAGONAL ELEVATED BUT WEAK - report the ratio and p-value;")
    print("  put depth-ordering on A3's rising-vs-decaying curves.")
else:
    print("  NO CLEAN DIAGONAL - the trained-vs-random contrast is")
    print("  unaffected and remains the headline.")

# ---- figure: raw matrix, depth trend, lift ----
fig = plt.figure(figsize=(14.2, 4.9))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.15, 1], wspace=0.30)

ax = fig.add_subplot(gs[0])
im = ax.imshow(M_t, vmin=0, vmax=1, cmap="magma", origin="lower")
ax.plot(argmax, idx, "o--", c="#39d0c6", lw=1.5, ms=4,
        label="measured best match")
ax.plot(idx, idx, ":", c="white", lw=1.2, alpha=0.75,
        label="exact diagonal")
ax.set_xlabel("Pythia layer"); ax.set_ylabel("GPT-2 layer")
ax.set_title(f"A - measured matrix\nrow means vary "
             f"{rm.min():.2f}-{rm.max():.2f} (hubness)", fontsize=9.8)
ax.legend(fontsize=7, loc="lower right", labelcolor="white",
          facecolor="#00000055", edgecolor="none")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

ax = fig.add_subplot(gs[1])
ax.plot(idx, diag, "o-", c="#1a5276", lw=2, ms=5, label="CKA diagonal")
ax.plot(idx[lo:hi + 1], diag[lo:hi + 1], "o-", c="#c0392b", lw=3, ms=6,
        zorder=5, label=f"L{lo}-L{hi}: rho={rho:.2f}")
ax.set_xlabel("layer"); ax.set_ylabel("CKA on the diagonal")
ax.set_title(f"B - depth trend\n{diag[lo]:.3f} -> {diag[hi]:.3f}, "
             f"p = {pv:.1e}", fontsize=9.8)
ax.legend(fontsize=7.5, frameon=False, loc="upper left")
ax.grid(alpha=0.25)

ax = fig.add_subplot(gs[2])
im = ax.imshow(lift, vmin=0.5, vmax=1.5, cmap="RdBu_r", origin="lower")
ax.plot(am_l, idx, "o--", c="#111", lw=1.4, ms=4,
        label="best match (marginals out)")
ax.plot(idx, idx, ":", c="#111", lw=1.2, alpha=0.6)
ax.set_xlabel("Pythia layer"); ax.set_ylabel("GPT-2 layer")
ax.set_title(f"C - marginals divided out\ndiag lift {dl.mean():.2f} vs "
             f"{odl:.2f}, p = {p_l:.3f}", fontsize=9.8)
ax.legend(fontsize=7, loc="lower right", frameon=False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

plt.tight_layout()
plt.savefig(str(DATA_DIR / "cka_real_structure.png"), dpi=150,
            bbox_inches="tight")
plt.show()
print("\nSaved cka_real_structure.png")